In [ ]:
!pip install -q unsloth unsloth_zoo
!pip install -q trl transformers datasets accelerate
# !pip install -q faker

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 114.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 86.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 101.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 122.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.1/225.1 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/2

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
max_seq_length = 1024

In [ ]:
from unsloth import FastLanguageModel

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(model_name,load_in_4bit=True,max_seq_length=max_seq_length)

==((====))==  Unsloth 2026.8.22: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth"
)

In [ ]:
from datasets import Dataset
import json
# import sys
# sys.path.append('/content/drive/MyDrive/text_extraction_ml')
# from synth_pipeline_v4_1 import generate_synth_data
from synth_pipeline_v5_2 import generate_synth_data


# Your existing structured generation setup
NUM_SAMPLES = 5000
data = []

EOS_TOKEN = tokenizer.eos_token # Essential to avoid infinite loops during inference

for _ in range(NUM_SAMPLES):
    example = generate_synth_data()
    text_structure = f"""### Instruction:
{example["instruction"]}

### Schema:
{json.dumps(example["schema"], ensure_ascii=False, indent=2)}

### Input:
{example["input"]}

### Response:
{json.dumps(example["output"], ensure_ascii=False, indent=2)}{EOS_TOKEN}"""

    data.append({"text": text_structure})

dataset = Dataset.from_list(data)
dataset = dataset.train_test_split(test_size=0.1, seed=42)

train_dataset = dataset["train"]
eval_dataset = dataset["test"]

In [ ]:
token_lengths = []

for i, sample in enumerate(train_dataset):
    tokens = tokenizer(
        sample["text"],
        add_special_tokens=True,
        truncation=False
    )
    token_lengths.append(len(tokens["input_ids"]))

print("Number of training records:", len(token_lengths))
print("Minimum tokens:", min(token_lengths))
print("Maximum tokens:", max(token_lengths))
print("Average tokens:", sum(token_lengths) / len(token_lengths))

print("\nRecords exceeding limits:")
print("Over 512:", sum(x > 512 for x in token_lengths))
print("Over 1024:", sum(x > 1024 for x in token_lengths))
print("Over 2048:", sum(x > 2048 for x in token_lengths))
print("Over 4096:", sum(x > 4096 for x in token_lengths))

Number of training records: 4500
Minimum tokens: 186
Maximum tokens: 908
Average tokens: 511.1173333333333

Records exceeding limits:
Over 512: 1997
Over 1024: 0
Over 2048: 0
Over 4096: 0


In [ ]:
# optionally save the raw dataset in jsonl

import json

with open("/content/drive/MyDrive/qwen2.5_1.5b_text_extraction_ml/synthetic_dataset.jsonl", "w", encoding="utf-8") as f:
    for item in data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

# Save train dataset
with open("/content/drive/MyDrive/qwen2.5_1.5b_text_extraction_ml/train_dataset.jsonl", "w", encoding="utf-8") as f:
    for row in train_dataset:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

# Save eval dataset
with open("/content/drive/MyDrive/qwen2.5_1.5b_text_extraction_ml/eval_dataset.jsonl", "w", encoding="utf-8") as f:
    for row in eval_dataset:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"Train samples: {len(train_dataset)}")
print(f"Eval samples: {len(eval_dataset)}")

Train samples: 4500
Eval samples: 500


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=1,
        num_train_epochs=3,

        learning_rate=2e-4,
        weight_decay=0.01,

        logging_steps=10,

        eval_strategy="epoch",
        # eval_strategy="steps",
        # eval_steps=10,


        save_strategy="no",
        # save_strategy="no",
        # save_steps=50,

        load_best_model_at_end=False,
        # load_best_model_at_end=False,

        fp16=True, # Changed from bf16=True to fp16=True
        report_to="none",
    ),
)

trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/4500 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/500 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,500 | Num Epochs = 3 | Total steps = 1,689
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Epoch,Training Loss,Validation Loss
1,0.499786,0.532504
2,0.498413,0.510874
3,0.456294,0.503620


TrainOutput(global_step=1689, training_loss=0.530607818075014, metrics={'train_runtime': 5793.5674, 'train_samples_per_second': 2.33, 'train_steps_per_second': 0.292, 'total_flos': 6.328228309561344e+16, 'train_loss': 0.530607818075014, 'epoch': 3.0})

In [ ]:
model.save_pretrained("/content/drive/MyDrive/qwen2.5_1.5b_text_extraction_ml/lora_model")
tokenizer.save_pretrained("/content/drive/MyDrive/qwen2.5_1.5b_text_extraction_ml/lora_model")

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/qwen2.5_1.5b_text_extraction_ml/lora_model/tokenizer_config.json.


('/content/drive/MyDrive/qwen2.5_1.5b_text_extraction_ml/lora_model/tokenizer_config.json',
 '/content/drive/MyDrive/qwen2.5_1.5b_text_extraction_ml/lora_model/chat_template.jinja',
 '/content/drive/MyDrive/qwen2.5_1.5b_text_extraction_ml/lora_model/tokenizer.json')

# Another round of training using the previously trained model

In [ ]:
from unsloth import FastLanguageModel

# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name="lora_model",  # path to saved adapter
#     max_seq_length=2048,
#     load_in_4bit=True,
# )

model_name = "/content/drive/MyDrive/qwen2.5_1.5b_text_extraction_ml/lora_model"

model, tokenizer = FastLanguageModel.from_pretrained(model_name=model_name,load_in_4bit=True,max_seq_length=max_seq_length)

==((====))==  Unsloth 2026.8.22: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [ ]:
# Generate another set of synthesized dataset to train the model with

import sys
sys.path.append('/content/drive/MyDrive/qwen2.5_1.5b_text_extraction_ml')
# from synth_pipeline_v4_1 import generate_example
from synth_pipeline_v5_2 import generate_example
from datasets import Dataset
import json

# Your existing structured generation setup
NUM_SAMPLES = 3000
data = []

EOS_TOKEN = tokenizer.eos_token # Essential to avoid infinite loops during inference

for _ in range(NUM_SAMPLES):
    example = generate_example()
    text_structure = f"""### Instruction:
{example["instruction"]}

### Schema:
{json.dumps(example["schema"], ensure_ascii=False, indent=2)}

### Input:
{example["input"]}

### Response:
{json.dumps(example["output"], ensure_ascii=False, indent=2)}{EOS_TOKEN}"""

    data.append({"text": text_structure})

dataset = Dataset.from_list(data)
dataset = dataset.train_test_split(test_size=0.1, seed=42)

train_dataset = dataset["train"]
eval_dataset = dataset["test"]

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=1,
        num_train_epochs=3,

        learning_rate=1e-4,
        weight_decay=0.01,

        logging_steps=10,

        eval_strategy="epoch",
        eval_steps=10,

        save_strategy="no",
        # save_strategy="no",
        # save_steps=50,

        load_best_model_at_end=False,
        # load_best_model_at_end=False,

        fp16=True, # Changed from bf16=True to fp16=True
        report_to="none",
    ),
)

trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/2700 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/300 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,700 | Num Epochs = 3 | Total steps = 1,014
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Epoch,Training Loss,Validation Loss
1,0.502053,0.505828
2,0.460778,0.501012
3,0.487915,0.499998


TrainOutput(global_step=1014, training_loss=0.48262684267653516, metrics={'train_runtime': 3456.1437, 'train_samples_per_second': 2.344, 'train_steps_per_second': 0.293, 'total_flos': 3.789415415786496e+16, 'train_loss': 0.48262684267653516, 'epoch': 3.0})

In [ ]:
model.save_pretrained("/content/drive/MyDrive/qwen2.5_1.5b_text_extraction_ml/lora_model_v2")
tokenizer.save_pretrained("/content/drive/MyDrive/qwen2.5_1.5b_text_extraction_ml/lora_model_v2")

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/qwen2.5_1.5b_text_extraction_ml/lora_model_v2/tokenizer_config.json.


('/content/drive/MyDrive/qwen2.5_1.5b_text_extraction_ml/lora_model_v2/tokenizer_config.json',
 '/content/drive/MyDrive/qwen2.5_1.5b_text_extraction_ml/lora_model_v2/chat_template.jinja',
 '/content/drive/MyDrive/qwen2.5_1.5b_text_extraction_ml/lora_model_v2/tokenizer.json')

In [ ]:
# final model with everything included for production use
model.save_pretrained_merged("/content/drive/MyDrive/qwen2.5_1.5b_text_extraction_ml/merged_model",tokenizer,save_method="merged_16bit")

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/qwen2.5_1.5b_text_extraction_ml/merged_model/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:49<00:00, 49.20s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:51<00:00, 111.59s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/qwen2.5_1.5b_text_extraction_ml/merged_model`


In [ ]:
# save gguf version for use in llm clients
model.save_pretrained_gguf("/content/drive/MyDrive/qwen2.5_1.5b_text_extraction_ml/gguf_model",tokenizer,quantization_method="f16")

Unsloth: Merging model weights to 16-bit format...


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/qwen2.5_1.5b_text_extraction_ml/gguf_model/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

...

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [01:05<00:00, 65.40s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [02:04<00:00, 124.55s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/qwen2.5_1.5b_text_extraction_ml/gguf_model`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Single-pass export: converting straight to ['f16'] - no separate quantize step.
 "-____-"     In total, you will have to wait at least 6 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b10687-mix-67dfc8b (app-b10687-mix-67dfc8b-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/content/drive/MyDrive/qwen2.5_1.5b_text_extraction_ml/gguf_model_gguf/qwen2.5-1.5b-instruct.F16.gguf']
Unsloth: Model files cleanup...
Unslot

{'save_directory': '/content/drive/MyDrive/qwen2.5_1.5b_text_extraction_ml/gguf_model',
 'gguf_directory': '/content/drive/MyDrive/qwen2.5_1.5b_text_extraction_ml/gguf_model_gguf',
 'gguf_files': ['/content/drive/MyDrive/qwen2.5_1.5b_text_extraction_ml/gguf_model_gguf/qwen2.5-1.5b-instruct.F16.gguf'],
 'modelfile_location': '/content/drive/MyDrive/qwen2.5_1.5b_text_extraction_ml/gguf_model_gguf/Modelfile',
 'want_full_precision': True,
 'is_vlm': False,
 'fix_bos_token': False}